In [1]:
import pandas as pd
from pathlib import Path

In [2]:
DATA_DIR = Path("..") / "data" / "processed"
combined_path = DATA_DIR / "combined_raw.csv"

df = pd.read_csv(combined_path)
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (308790, 3)


,peptide,source,label
0,AAAAAIFVI,IEDB,1
1,AAAAALDKKQRNFDKILA,IEDB,1
2,AAAAKLAGLVFPQPPAPIAV,IEDB,1
3,AAAGFASKTPANQAISMIDG,IEDB,1
4,AAAIFMTATPPGTAD,IEDB,1


In [4]:
print(f"Total peptides: {len(df):,}")
print(f"Columns: {list(df.columns)}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print("\nColumn types:")
print(df.dtypes)

Total peptides: 308,790
Columns: ['peptide', 'source', 'label']

Missing values:
peptide    0
source     0
label      0
dtype: int64

Column types:
peptide    object
source     object
label       int64
dtype: object


In [ ]:
Class distribution

In [9]:
class_counts = df["label"].value_counts().sort_index()
class_pct = df["label"].value_counts(normalize=True).sort_index() * 100

print(f"Label 0 (non-immunogenic): {class_counts[0]:,} ({class_pct[0]:.2f}%)")
print(f"Label 1 (immunogenic):     {class_counts[1]:,} ({class_pct[1]:.2f}%)")
print(f"\nClass imbalance ratio (0:1): {class_counts[0] / class_counts[1]:.2f}:1")

Label 0 (non-immunogenic): 220,235 (71.32%)
Label 1 (immunogenic):     88,555 (28.68%)

Class imbalance ratio (0:1): 2.49:1


Peptide length stats

In [8]:
df["peptide_length"] = df["peptide"].str.len()

print(df["peptide_length"].describe().round(2))

mhc_range = df["peptide_length"].between(8, 11)
print(f"\nPeptides with length 8–11: {mhc_range.sum():,} ({mhc_range.mean() * 100:.2f}%)")

print("\nMost common lengths:")
print(df["peptide_length"].value_counts().sort_index().head(20))

count    308790.00
mean         15.61
std           5.55
min           2.00
25%          11.00
50%          15.00
75%          18.00
max         142.00
Name: peptide_length, dtype: float64

Peptides with length 8–11: 80,357 (26.02%)

Most common lengths:
peptide_length
2          2
3          5
4          9
5         40
6        170
7        203
8       6336
9      46072
10     20999
11      6950
12      5029
13      3601
14      4066
15    117958
16      7192
17      6298
18     15870
19      2811
20     24314
21      2027
Name: count, dtype: int64


Verifying the data for duplicates or invalid peptides

In [10]:
# Missing / empty peptides
missing = df["peptide"].isna() | (df["peptide"].str.strip() == "")
print(f"Missing or empty peptides: {missing.sum()}")

# Duplicates
dup_mask = df["peptide"].duplicated()
print(f"Duplicate peptide sequences: {dup_mask.sum()}")

if dup_mask.sum() > 0:
    print("\nExamples of duplicate sequences (peptide, label, source):")
    print(df[dup_mask][["peptide", "label", "source"]].head(5).to_string(index=False))

# Only use the 20 standard amino acids 
standard_aa = set("ACDEFGHIKLMNPQRSTVWY")

def contains_invalid(seq):
    seq = str(seq).upper()
    return any(ch not in standard_aa for ch in seq)

invalid_mask = df["peptide"].apply(contains_invalid)
print(f"\nPeptides with other characters: {invalid_mask.sum()}")

if invalid_mask.sum() > 0:
    sample_invalid = df[invalid_mask].head(5)
    print("\nExample invalid sequences:")
    print(sample_invalid[["peptide", "label", "source"]])

    problematic_chars = sorted({
        ch
        for seq in df[invalid_mask]["peptide"]
        for ch in seq.upper()
        if ch not in standard_aa
    })
    print(f"\nCharacters outside the standard 20 AA: {problematic_chars}")

Missing or empty peptides: 0
Duplicate peptide sequences: 81091

Examples of duplicate sequences (peptide, label, source):
           peptide  label source
         AAAAAIFVI      0   IEDB
AAAAALDKKQRNFDKILA      0   IEDB
   AAAIFMTATPPGTAD      0   IEDB
   AAALEQLLGQTADVA      0   IEDB
  AAASAIQGNVTSIHSL      0   IEDB

Peptides with other characters: 2359

Example invalid sequences:
                                    peptide  label source
339                 AGTVNIGASDAY + PALM(A1)      1   IEDB
464                           AKXVAAWTLKAAA      1   IEDB
1042  AVLTGYSLFQKEKMVLNEGTS + MCM(V15, L16)      1   IEDB
1068                AVWRIDTPDKLT + ACET(A1)      1   IEDB
1077             AVYTRIMMNGGRLKR + GLYC(T4)      1   IEDB

Characters outside the standard 20 AA: [' ', '(', ')', '+', ',', '-', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'B', 'O', 'U', 'X']
